<a href="https://colab.research.google.com/github/NataliaPinkoff/Travel-Agent-AI/blob/main/TravelAgentAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:
#@title Configuração Inicial
# Instalação limpa apenas do necessário
!pip install -q google-generativeai==0.3.2

import google.generativeai as genai
from google.colab import userdata

#@title Configuração da API Gemini
try:
  API_KEY = userdata.get('GEMINI_API_KEY')
except:
  API_KEY = "" #@param {type:"string"}
  if not API_KEY:
    raise ValueError("Por favor, insira sua chave de API do Google AI Studio")

# Configuração atualizada do Gemini
genai.configure(api_key=API_KEY)

# Usando o modelo mais recente
model = genai.GenerativeModel('gemini-1.5-pro-latest')  # Modelo atualizado

#@title Agente de Planejamento de Viagens
class TravelAgent:
    def __init__(self):
        self.model = model

    def pesquisar_atracoes(self, cidade, interesse=""):
        prompt = f"""
        Você é um especialista em turismo. Liste os principais pontos turísticos em {cidade}
        {f"com foco em {interesse}" if interesse else ""}.
        Para cada atração forneça:
        - Nome
        - Descrição breve
        - Tempo médio de visita
        - Melhor horário para visita

        Formate a resposta como:
        • [Nome] | [Tempo] | [Horário ideal]
          → [Descrição]
        """
        try:
            response = self.model.generate_content(prompt)
            return response.text
        except Exception as e:
            print(f"Erro na pesquisa: {str(e)}")
            return None

    def criar_roteiro(self, cidade, dias, atracoes, interesse=""):
        prompt = f"""
        Você é um planejador de viagens profissional. Crie um roteiro para {dias} dias em {cidade}.

        Atrações disponíveis: {atracoes[:3000]}...

        Requisitos:
        1. Organize por proximidade geográfica
        2. Balanceie atividades diárias
        3. {f"Destaque {interesse}" if interesse else "Variedade de experiências"}
        4. Inclua sugestões de restaurantes
        5. Seja realista com tempos

        Formato:
        Dia X:
        ☀ Manhã:
          - [Atividade] ([horário])
            ○ [Detalhes]
            ○ [Tempo estimado]
        🌞 Tarde:
          - [Atividade] ([horário])
            ○ [Onde almoçar/jantar]
        """
        try:
            response = self.model.generate_content(prompt)
            return response.text
        except Exception as e:
            print(f"Erro no planejamento: {str(e)}")
            return None

#@title Interface Principal
def gerar_roteiro_completo(cidade, dias, interesse=""):
    print(f"✈ Planejando sua viagem para {cidade} ({dias} dias)...\n")

    agent = TravelAgent()

    print("🔍 Buscando as melhores atrações...")
    atracoes = agent.pesquisar_atracoes(cidade, interesse)

    if not atracoes:
        print("❌ Falha ao buscar atrações")
        return None

    print("\n📅 Criando seu roteiro personalizado...")
    roteiro = agent.criar_roteiro(cidade, dias, atracoes, interesse)

    return roteiro

#@title Execução
cidade = "Curitiba" #@param {type:"string"}
dias = 15 #@param {type:"integer", min:1, max:10}
interesse = "história e cultura" #@param ["", "história e cultura", "praias e natureza", "compras e gastronomia", "aventura"]

print("⏳ Iniciando planejamento...\n")
try:
    resultado = gerar_roteiro_completo(cidade, dias, interesse)

    if resultado:
        print("\n" + "="*60)
        print("🌟 SEU ROTEIRO DE VIAGEM 🌟".center(60))
        print("="*60 + "\n")
        print(resultado)

        # Salvar em arquivo
        nome_arquivo = f"roteiro_{cidade.replace(' ', '_')}_{dias}dias.txt"
        with open(nome_arquivo, "w") as f:
            f.write(resultado)
        print(f"\n✅ Roteiro salvo como '{nome_arquivo}'")
    else:
        print("\n❌ Não foi possível gerar o roteiro completo")

except Exception as e:
    print("\n⚠️ Ocorreu um erro inesperado:")
    print(f"- {str(e)}")
    print("\n🔍 O que fazer:")
    print("1. Verifique sua chave de API no Google AI Studio")
    print("2. Confira se o modelo 'gemini-1.5-pro-latest' está disponível")
    print("3. Tente reduzir o número de dias ou atrações")

⏳ Iniciando planejamento...

✈ Planejando sua viagem para Curitiba (15 dias)...

🔍 Buscando as melhores atrações...

📅 Criando seu roteiro personalizado...

                 🌟 SEU ROTEIRO DE VIAGEM 🌟                  

## Roteiro de 15 dias em Curitiba:

**Dia 1:**

☀ Manhã:
  - Chegada em Curitiba e check-in no hotel (horário flexível)
    ○ Sugestão de hotel: próximo ao centro histórico para facilitar os deslocamentos dos primeiros dias.
    ○ Tempo estimado: 1-2 horas (dependendo do transporte do aeroporto)

🌞 Tarde:
  - Setor Histórico (14:00)
    ○ Explorar a Rua São Francisco, Catedral Basílica, Praça Tiradentes.
    ○ Tempo estimado: 3 horas
    ○ Onde almoçar: Bar do Alemão (comida típica alemã) ou Restaurante Velhos Tempos (comida brasileira).

**Dia 2:**

☀ Manhã:
  - Paço da Liberdade (09:00)
    ○ Visita ao Museu da Imagem e do Som.
    ○ Tempo estimado: 2 horas

🌞 Tarde:
  - Casa Romário Martins (14:00)
    ○ Conhecer a vida e obra do escritor.
    ○ Tempo estimado: 1 hora